# Reproduce — Plain Q-Former (nhánh `main`)

Notebook tái lập **từ đầu**, không nạp lại checkpoint cũ.

**Kiến trúc:** MF (đông cứng) → Q-Former BLIP-2 → `llm_proj` → Vicuna-7B-v1.5 (đông cứng + LoRA).
Q-Former hiện thực trên `transformers.InstructBlipQFormerModel` nhưng **chạy đúng đường BLIP-2**:
truy vấn chỉ cross-attention vào vector CF, **không có instruction nào đi vào đường nuôi LLM**.
Nhánh text của Q-Former chỉ sống trong Stage-1 (ITC/ITM/ITG) với metadata thật của phim.

**Ba nhánh khảo sát (arm):**

| Arm | Ý nghĩa | Cách bật |
|---|---|---|
| `B0-textonly` | Không tiêm CF, chỉ prompt chữ | đánh giá checkpoint R3 với `--step 1` |
| `B2-qformer` | Q-Former, **truy vấn dùng chung** (BLIP-2 nguyên bản) | `user_conditioned=False` |
| `B3-qformer-ucq` | B2 + **truy vấn điều kiện theo người dùng** ← kết quả chính | `user_conditioned=True` (mặc định) |

Tên arm được nội suy thẳng vào `output_dir`, nên hai arm **không đè checkpoint của nhau**.

**Giao thức:** giữ nguyên CoLLM — `match_sella_history_filter=false`, test tách 3 nhánh
`test / test_warm / test_cold`. Chọn checkpoint theo **uAUC** (CoLLM/CoRA chọn theo AUC — khác biệt
này phải nêu trong luận). Chi tiết ở khối chú thích đầu `configs/config.yaml`.

# 0. Môi trường

## 0.1 Clone

Repo trên GitHub tên `CoQLLM`, nhưng **phải clone vào thư mục `/content/CoQLLM`** — toàn bộ
đường dẫn trong `configs/config.yaml` (data, ckpt, prompts) hardcode tiền tố `/content/CoQLLM/`.

In [ ]:
import os
from getpass import getpass

token = getpass("GitHub Personal Access Token (PAT): ")

!git clone https://{token}@github.com/QuocBaoBuiNguyen/CoQLLM.git /content/CoQLLM
%cd /content/CoQLLM
!git checkout main
!git log --oneline -5

In [ ]:
# Chạy lại ô này mỗi khi push code mới lên nhánh.
!git -C /content/CoQLLM pull origin main

## 0.2 Nạp lại module + PYTHONPATH

In [ ]:
import sys
import importlib
from types import ModuleType

# Vá `imp` (bị gỡ khỏi Python 3.12) cho các thư viện cũ còn import nó.
imp = ModuleType('imp')
imp.reload = importlib.reload
sys.modules['imp'] = imp

In [ ]:
%env PYTHONPATH=src:$PYTHONPATH

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 0.3 Miniconda

In [ ]:
!wget -c https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /content/Miniconda3.sh
!chmod +x /content/Miniconda3.sh
!bash /content/Miniconda3.sh -b -f -p /usr/local

In [ ]:
import sys
sys.path.append('/usr/local/lib/python3.12/site-packages')

!conda --version
!source /usr/local/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!source /usr/local/etc/profile.d/conda.sh && conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

## 0.4 Khôi phục env `sigllm` từ cache trên Cloudflare R2

Không dựng bằng `conda env create -f env.yaml` — dựng từ đầu mất ~20 phút và `env.yaml`
đang pin `transformers==4.28.0` (bản đó **chưa có** `InstructBlipQFormerModel`, code sẽ không
import nổi). Cache tarball là môi trường thật đã dùng cho mọi kết quả.

In [ ]:
!pip install -q "docutils>=0.20,<0.22"
!pip install -q awscli

In [ ]:
import os
from google.colab import drive, userdata

drive.mount('/content/drive', force_remount=True)

os.environ['AWS_ACCESS_KEY_ID'] = userdata.get('R2_ACCESS_KEY_ID')
os.environ['AWS_SECRET_ACCESS_KEY'] = userdata.get('R2_SECRET_ACCESS_KEY')
os.environ['AWS_DEFAULT_REGION'] = 'auto'

CLOUDFLARE_ACCOUNT_ID = "b712ab0bb7699d3a04331f27f94ff98f"
os.environ['ENDPOINT_URL'] = f"https://{CLOUDFLARE_ACCOUNT_ID}.r2.cloudflarestorage.com"

In [ ]:
%%bash
set -e
ENV_NAME="sigllm"

aws --endpoint-url="$ENDPOINT_URL" s3 cp "s3://sigllm/colab_conda_cache/${ENV_NAME}.tar.gz" "/content/${ENV_NAME}.tar.gz"

mkdir -p "/usr/local/envs/${ENV_NAME}"
tar -xzf "/content/${ENV_NAME}.tar.gz" -C "/usr/local/envs/${ENV_NAME}"
rm -f "/content/${ENV_NAME}.tar.gz"

source /usr/local/etc/profile.d/conda.sh
conda activate "$ENV_NAME"

## 0.5 Chốt phiên bản — **môi trường của hồ sơ**

`transformers==4.38.2` là bản đã sinh ra mọi số trong luận. Đừng nâng lên:
từ 4.45 HuggingFace sửa `Blip2QFormerModel` và đổi chữ ký `query_length`, hành vi sẽ lệch.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
python -m pip install -U "transformers==4.38.2" "accelerate==0.27.2"

In [ ]:
# Ghi lại phiên bản để dán vào bảng "môi trường thực nghiệm" của luận.
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
pip list | grep -Ei "^(transformers|peft|accelerate|torch|tokenizers|bitsandbytes|scikit-learn) "

# 1. Chuẩn bị dữ liệu (ML-1M)

In [ ]:
import sys
sys.path.insert(0, "src")

# Xoá sạch để đảm bảo chạy lại từ đầu, không lẫn dữ liệu cũ.
!rm -rf /content/CoQLLM/data/processed/
!unzip -o -q /content/CoQLLM/data/raw/ml-1m.zip -d /content/CoQLLM/data/raw/
!ls /content/CoQLLM/data/raw/ml-1m

## 1.1 Tiền xử lý chính

Tách theo thời gian, ánh xạ lại ID, dựng lịch sử tương tác tuần tự.
**Giữ toàn bộ dòng và đệm 0 cho lịch sử ngắn** (`match_sella_history_filter=false`) — đây là
điểm bám giao thức CoLLM; bật `true` sẽ loại các dòng lịch sử < 2 và làm tập test dễ đi.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/datasets/data_preprocessing.py

## 1.2 Gắn nhãn warm / strict-cold cho tập test

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/datasets/preprocess_test_cold_warm.py

## 1.3 Dựng pickle cho Q-Former

Sinh `train/valid/test_qformer_ood2.pkl` — cặp *item ↔ metadata* mà Stage-1 dùng cho
ITC/ITM/ITG, và cặp item-item / user-item cho hai loss cộng tác. **Bắt buộc chạy trước R1.**

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/build_qformer_dataset.py \
  --cfg-path /content/CoQLLM/configs/config.yaml

# 2. Trọng số LLM — Vicuna-7B-v1.5

Chép từ Drive (nhanh). Nếu chưa có trên Drive thì dùng ô `pull_llm_model.py` bên dưới để tải
từ HuggingFace rồi tự sao lưu lại.

In [ ]:
!mkdir -p /content/CoQLLM/ckpt/llm/base/
!cp -r /content/drive/MyDrive/CoQLLM/llm/vicuna-7b/* /content/CoQLLM/ckpt/llm/base/
!ls /content/CoQLLM/ckpt/llm/base/

In [ ]:
# !source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
# python /content/CoQLLM/src/coqllm/pipelines/llm/pull_llm_model.py

# 3. Huấn luyện

Thứ tự bắt buộc. **R0–R3 dùng chung cho mọi arm**; chỉ R4/R5 là rẽ nhánh khảo sát.

| | Bước | Huấn luyện cái gì | Arm |
|---|---|---|---|
| R0 | MF | embedding user/item cộng tác, sau đó **đông cứng** | chung |
| R1 | Stage-1 | Q-Former: ITC + ITM + ITG + item-item + user-item | chung |
| R2 | Stage-2 | Q-Former + `llm_proj`, mục tiêu sinh văn bản | chung |
| R3 | Stage-3 Step-1 | **chỉ LoRA**, prompt thuần chữ (Q-Former đóng băng) | → `B0-textonly` |
| R5 | Stage-3 Step-2 | Q-Former + `llm_proj`, prompt đầy đủ, LoRA đóng băng | `B3-qformer-ucq` |
| R4 | Stage-3 Step-2 | như trên, tắt điều kiện người dùng | `B2-qformer` |

## R0 — MF (nền cộng tác)

Sinh `e_u`, `e_i`. Mọi bước sau chỉ **đọc** MF, không cập nhật nó.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/rec/train_rec_baseline.py \
  --cfg-path /content/CoQLLM/configs/config.yaml

## R1 — Stage-1: căn chỉnh biểu diễn

Dạy Q-Former biến vector CF thành 8 soft token gắn được với ngữ nghĩa văn bản.
Năm mục tiêu: ITC (đối lập item↔text), ITM (khớp nhị phân, âm khó lấy từ ma trận ITC),
ITG (sinh văn bản có điều kiện), cộng item-item và user-item.

Đây là bước **chỉ Q-Former làm được** — một MLP không có nhánh text nên ITM/ITG không định
nghĩa được. Điểm này đáng nêu trong luận.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage1_representation.py \
  --cfg-path /content/CoQLLM/configs/config.yaml

## R2 — Stage-2: tiền huấn luyện sinh

Nối Q-Former vào LLM đông cứng và huấn luyện `llm_proj` bằng mục tiêu ngôn ngữ trên văn bản
item. Stage-2 gọi `encode_cf` (**không** có text) — trùng khớp với đường Stage-3 sau khi đã gỡ
instruction, nên `llm_proj` được huấn luyện và sử dụng trên cùng một phân phối đầu vào.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && conda activate sigllm && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage2_generative.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.qformer_stage2.lr=2e-4 \
            run.qformer_stage2.weight_decay=1e-4 \
            run.qformer_stage2.early_stopping_patience=8

## R3 — Stage-3 Step-1: LoRA trên prompt thuần chữ

Theo đúng quy trình hai bước của CoLLM. Prompt **không có** `<ItemIDList>` / `<TargetItemID>`,
nên LoRA học riêng phần tác vụ Yes/No mà chưa thấy soft token nào.

Checkpoint của bước này **chính là baseline `B0-textonly`** — đánh giá nó ở mục 4.3.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step1_lora.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc

## R5 — Stage-3 Step-2, arm `B3-qformer-ucq` (kết quả chính)

Nạp LoRA tốt nhất của R3 và **đóng băng nó**, rồi chỉ tinh chỉnh Q-Former + `llm_proj` trên
prompt đầy đủ. `user_conditioned=True` (mặc định trong config) nên 8 truy vấn được dịch thêm
`user_proj(user_cf)`: cùng một bộ phim sẽ được đọc bằng góc nhìn riêng của từng người dùng.

Ghi vào `ckpt/qformer_stage3_step2_B3-qformer-ucq_vicuna/`.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc

## R4 — Stage-3 Step-2, arm `B2-qformer` (khảo sát)

Y hệt R5, **đổi đúng một biến**: truy vấn dùng chung cho mọi người dùng — tức Q-Former BLIP-2
nguyên bản. Không cần chạy lại R0–R3.

Hiệu số R5 − R4 chính là đóng góp của *truy vấn điều kiện theo người dùng*. Dự kiến AUC tăng rõ
còn uAUC gần như đứng yên: phép dịch cộng thêm **cố định trong một người dùng** nên triệt tiêu
khi xếp hạng nội-người-dùng, chỉ dịch được điểm giữa các người dùng.

Ghi vào `ckpt/qformer_stage3_step2_B2-qformer_vicuna/` — không đè lên R5.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python /content/CoQLLM/src/coqllm/pipelines/multimodal/train_qformer_stage3_step2_cie.py \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --options run.best_metric=uauc \
            model.qformer_config.user_conditioned=False \
            run.qformer_stage3_step2.arm=B2-qformer

# 4. Đánh giá

Cả ba mục dưới đều chấm trên `test / test_warm / test_cold` và in AUC + uAUC.

## 4.1 `B3-qformer-ucq` — kết quả chính

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --step 2

## 4.2 `B2-qformer`

Chỉ cần lặp lại đúng hai override của R4 — `output_dir` tự trỏ sang thư mục của arm này.

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --step 2 \
  --options model.qformer_config.user_conditioned=False \
            run.qformer_stage3_step2.arm=B2-qformer

## 4.3 `B0-textonly` — sàn không có tín hiệu cộng tác

Chấm thẳng checkpoint LoRA của R3 với `--step 1`: prompt thuần chữ, không soft token nào.
Đây là mức nền để trả lời "cầu nối CF đóng góp bao nhiêu".

In [ ]:
!source /usr/local/etc/profile.d/conda.sh && \
conda activate sigllm && \
export TOKENIZERS_PARALLELISM=false && \
python -m coqllm.pipelines.multimodal.eval_test \
  --cfg-path /content/CoQLLM/configs/config.yaml \
  --step 1

# 5. Sao lưu checkpoint

Colab ngắt phiên là mất sạch. Chạy sau **mỗi** bước R0–R5, đừng để dồn.

In [ ]:
!mkdir -p /content/drive/MyDrive/CoQLLM/plain-qformer/
!cp -r /content/CoQLLM/ckpt/mf                                        /content/drive/MyDrive/CoQLLM/plain-qformer/
!cp -r /content/CoQLLM/ckpt/qformer_stage1                            /content/drive/MyDrive/CoQLLM/plain-qformer/
!cp -r /content/CoQLLM/ckpt/qformer_stage2_vicuna                     /content/drive/MyDrive/CoQLLM/plain-qformer/
!cp -r /content/CoQLLM/ckpt/qformer_stage3_step1_lora_vicuna          /content/drive/MyDrive/CoQLLM/plain-qformer/
!cp -r /content/CoQLLM/ckpt/qformer_stage3_step2_B3-qformer-ucq_vicuna /content/drive/MyDrive/CoQLLM/plain-qformer/
!cp -r /content/CoQLLM/ckpt/qformer_stage3_step2_B2-qformer_vicuna     /content/drive/MyDrive/CoQLLM/plain-qformer/
!ls -la /content/drive/MyDrive/CoQLLM/plain-qformer/

In [ ]:
# Bản sao lên Cloudflare R2 (bucket MỚI `coqllm`; conda_cache vẫn ở bucket cũ `sigllm`).
!aws --endpoint-url="$ENDPOINT_URL" s3 sync /content/CoQLLM/ckpt/ s3://coqllm/ckpt/